## Setup


Before you run this notebook, you will need to configure the following:
- create a GitHub PAT
- add the following items to your Colab vault:
  - your GitHub PAT as `gh`
  - your GitHub username as `user_name`
  - your GitHub e-mail address as `user_email`

> Optional: use ssh to connect to GitHub
> - create an ssh key of type `ed25519`
> - upload the public key to GitHub
> - store the private key in Colab's vault as `ssh_private`



In [ ]:
# @title Install packages - Jupyter
%%capture
%%bash

# jupyter install
cat <<'eof' > /tmp/jupyterlab.install.sh
  pip install ipyaml jupyterlab
  jupyter labextension disable @jupyterlab/apputils-extension:announcements
  pip install bash_kernel
  python -m bash_kernel.install
  mkdir -p ~/.jupyter/lab/user-settings/\@jupyterlab/apputils-extension/
  echo '{ "theme": "JupyterLab Dark" }' >  ~/.jupyter/lab/user-settings/\@jupyterlab/apputils-extension/themes.jupyterlab-settings
  jupyter lab \
    --ip=127.0.0.1 \
    --port=8888 \
    --no-browser \
    --allow-root \
    --NotebookApp.token='' \
    ;
  echo == Done
  sleep 30
  rm -f "${0}"
eof

tmux new -s jupyter-server -d "bash /tmp/jupyterlab.install.sh"



In [ ]:
# @title Install packages - system
%%capture
%%bash

# package installs
cat <<'eof' > /tmp/installs.sh

  # install glow, a markdown renderer for the terminal
  dpkg -l glow || {
    mkdir -p /etc/apt/keyrings
    curl -fsSL https://repo.charm.sh/apt/gpg.key | gpg --dearmor -o /etc/apt/keyrings/charm.gpg
    echo "deb [signed-by=/etc/apt/keyrings/charm.gpg] https://repo.charm.sh/apt/ * *" | tee /etc/apt/sources.list.d/charm.list
    apt-get update && apt-get install -y glow
  }

  # update the package index
  apt-get update

  # install system packages ... quickly
  apt-get install -y \
    jq \
    less \
    ncal \
    screen \
    tree \
    zstd \
    ;

  # these packages take a while to install
  apt-get install -y \
    pandoc \
    texlive-xetex \
    ;

  # make /kaggle/input writeable
  mount -o remount,rw /kaggle/input/

  echo == Done
  sleep 30
  rm -f "${0}"
eof

tmux new -s update -d "bash /tmp/installs.sh"


In [ ]:
# @title Configure system environment
%%capture
%%bash

# bash setup
fgrep -q '.bash_aliases' ~/.bashrc || {
  echo -e '\n\n\n[ -f ~/.bash_aliases ] && . ~/.bash_aliases\n' >> ~/.bashrc
}

# Setup initial aliases
{ cat << 'eof'
alias cls=clear
alias dir='ls -la'

eof
} >> ~/.bash_aliases


In [ ]:
# @title Python modules, etc.
%alias tree tree

from datetime import datetime, timezone
from time import sleep
import requests
import os
from google.colab import output
from google.colab import userdata

# Get authentication info
user_email = userdata.get('user_email')
user_name = userdata.get('user_name')
gh = userdata.get('gh')
! echo export GH_CLI_TOKEN={gh} >> ~/.bash_aliases
! echo export GH_TOKEN={gh} >> ~/.bash_aliases

# Set authentication info for ssh
# ssh_private = userdata.get('ssh_private').replace('\r', '')
# filepath = '/root/.ssh/id_ed25519'
# with open( filepath, 'w' ) as f:
#   f.write( ssh_private )
# os.chmod( filepath, 0o600 )

! git config --global user.email "{user_email}"
! git config --global user.name "{user_name}"
! ssh-keyscan -t ed25519,rsa github.com > ~/.ssh/known_hosts 2> /dev/null


In [ ]:
# @title Setup GitHub environment
%%bash

cat <<'eof' > /tmp/github.setup.sh
  cd /content/
  mkdir -p src
  cd src

  # Clone GitHub repos
  git clone https://github.com/rwcitek/job-sites-and-listings.git
  echo == Done
  sleep 10
  rm -f "${0}"
eof

tmux new -s github -d "bash /tmp/github.setup.sh"


In [ ]:
# @title Wait for Jupyter to start
print("Waiting for Jupyter to start")
for i in range(300):
  try:
    requests.head( "http://127.0.0.1:8888" )
    print()
    break
  except:
    print("=", end="")
  sleep(1)
print(f"{i} seconds")

print("Jupyter is ready.")
output.serve_kernel_port_as_window(8888)


In [ ]:
# @title Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# @title Timer for 'h' hours
h = 10    # number of hours
print("Timer 2")
print(f"Started at ... \n{datetime.now(timezone.utc).timetz().isoformat(timespec='seconds')}")

for i in range(60*60*h):
  utc_now = datetime.now(timezone.utc)
  print(f"\r{utc_now.timetz().isoformat(timespec='seconds')} {'==' * (utc_now.second % 10)}", end='')
  sleep(1)
